# ShopStream Data Exploration

Quick analysis of the pipeline output layers.

In [ ]:
import duckdb
import pandas as pd
import os

con = duckdb.connect()
con.execute("""
    INSTALL httpfs; LOAD httpfs;
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

In [ ]:
daily_revenue = con.execute("""
    SELECT * FROM delta_scan('s3://shopstream/gold/fct_daily_revenue')
    ORDER BY order_date DESC
    LIMIT 30
""").df()
daily_revenue

In [ ]:
funnel = con.execute("""
    SELECT * FROM delta_scan('s3://shopstream/gold/fct_conversion_funnel')
    ORDER BY date_partition DESC
    LIMIT 14
""").df()
funnel

In [ ]:
customers = con.execute("""
    SELECT customer_tier, COUNT(*) as count, ROUND(AVG(lifetime_value), 2) as avg_ltv
    FROM delta_scan('s3://shopstream/gold/dim_customers')
    GROUP BY customer_tier
    ORDER BY avg_ltv DESC
""").df()
customers

In [ ]:
order_status = con.execute("""
    SELECT status, COUNT(*) as count, ROUND(SUM(total_amt), 2) as revenue
    FROM delta_scan('s3://shopstream/silver/orders')
    GROUP BY status
    ORDER BY count DESC
""").df()
order_status